# DEXPI ingestion — step 1: understand the source and graph

Run **one code cell at a time** with **Shift+Enter**. Read the explanation before each cell.

We will inspect the XML, load the same file with **pyDEXPI**, and examine the resulting graph. We do not call an LLM, create embeddings or write to LightRAG in this notebook.

The XML checks and graph checks answer different questions:
- **XML:** what is explicitly present in the source file?
- **pyDEXPI:** what has the library represented as nodes, attributes and edges?

Place this notebook in your project root. Keep the XML under `data/documents/`. Select the kernel **Python (raganything-hazop)**.

Variables remain in memory between cells. If you change an earlier cell, rerun the dependent cells below it. Use **Restart Kernel** to start fresh.


## 1. Check which Python is running

The notebook's **kernel** is its Python process. It must use your project environment, otherwise imports can fail even when the packages are installed elsewhere.

`sys.executable` shows that process's Python path. `version(...)` checks the installed package versions.


In [74]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.executable)
for package in ("pydexpi", "networkx", "lightrag-hku", "raganything"):
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "not installed in this kernel")


Python: /home/duc/Desktop/raganything_hazop/.venv/bin/python
pydexpi 1.1.0
networkx 3.6.1
lightrag-hku 1.4.16
raganything 1.4.1


## 2. Locate the XML

`Path.cwd()` is the notebook's working directory. The `/` operator joins path components; it does not divide numbers here.

If the file is not found, change `PROJECT_ROOT` to the actual project directory, for example `Path(r"C:\Users\gauva\Desktop\raganything-hazop")`.


In [75]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
XML_PATH = PROJECT_ROOT / "data" / "others" / "C01V04-VER.EX01.xml"

print("Project:", PROJECT_ROOT)
print("XML:", XML_PATH)
assert XML_PATH.is_file(), "XML not found. Check PROJECT_ROOT and the filename."


Project: /home/duc/Desktop/raganything_hazop
XML: /home/duc/Desktop/raganything_hazop/data/others/C01V04-VER.EX01.xml


## 3. Parse the XML tree

Python's `ElementTree` parses the XML syntax. **It does not create the engineering knowledge graph.**

`root` represents the outermost element. Each element has a tag, attributes and child elements. We reject DTD/entity declarations because this exercise does not need them.

A variable on the last line of a notebook code cell is displayed automatically.


In [76]:
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

xml_bytes = XML_PATH.read_bytes()
if b"<!DOCTYPE" in xml_bytes.upper() or b"<!ENTITY" in xml_bytes.upper():
    raise ValueError("DTD/entity declarations are not supported in this exercise")
root = ET.fromstring(xml_bytes)

{"root_tag": root.tag, "root_attributes": root.attrib}


{'root_tag': 'PlantModel', 'root_attributes': {}}

## 4. Exclude the shape catalogue from the source inventory

The shape catalogue contains drawing templates. Counting these as installed equipment would be misleading.

`local_name()` removes an XML namespace prefix such as `{namespace}`. The stack walks the tree while remembering whether an element is inside `ShapeCatalogue`. This filter is for our XML inventory; it does not modify the XML passed to pyDEXPI.


In [77]:
def local_name(tag):
    return tag.rsplit("}", 1)[-1]

live_elements = []
parents = {}
stack = [(root, False, None)]
catalogue_element_count = 0

while stack:
    element, inside_catalogue, parent = stack.pop()
    inside_catalogue = inside_catalogue or local_name(element.tag) == "ShapeCatalogue"
    parents[element] = parent
    if inside_catalogue:
        catalogue_element_count += 1
    else:
        live_elements.append(element)
    for child in reversed(element):
        stack.append((child, inside_catalogue, element))

tag_counts = Counter(local_name(element.tag) for element in live_elements)
{"catalogue_elements_excluded": catalogue_element_count,
 "selected_live_element_counts": {name: tag_counts[name] for name in
     ("Equipment", "Nozzle", "PipingComponent", "PipingNetworkSegment",
      "InformationFlow", "Connection", "ConnectionPoints", "Node")}}


{'catalogue_elements_excluded': 459,
 'selected_live_element_counts': {'Equipment': 16,
  'Nozzle': 19,
  'PipingComponent': 19,
  'PipingNetworkSegment': 23,
  'InformationFlow': 6,
  'Connection': 29,
  'ConnectionPoints': 47,
  'Node': 148}}

## 5. Inspect one equipment item and its specifications

Select the first live `Equipment` element. Its `ID` is a source identifier; `ComponentClass` describes its type.

Only direct `GenericAttributes` children are shown here, so we do not accidentally mix a nozzle's properties into its parent equipment.

**Try:** change `[0]` to `[1]` to inspect another equipment item.


In [78]:
equipment = [e for e in live_elements if local_name(e.tag) == "Equipment"]
assert equipment, "No live Equipment elements found"
selected_equipment = equipment[0]

from pprint import pprint
print("-------------- Equipment attributes: ------------------")
pprint(dict(selected_equipment.attrib))

attributes = []
print("------------ Generic Attributes --------------" )
for child in selected_equipment:
    if local_name(child.tag) == "GenericAttributes":
        attributes.extend(dict(a.attrib) for a in child
                          if local_name(a.tag) == "GenericAttribute")
pprint(attributes)


-------------- Equipment attributes: ------------------
{'ComponentClass': 'PlateHeatExchanger',
 'ComponentClassURI': 'http://sandbox.dexpi.org/rdl/PlateHeatExchanger',
 'ComponentName': 'PLATE_TYPE_HEAT_EXCHANGER_SHAPE',
 'ID': 'PlateHeatExchanger-1'}
------------ Generic Attributes --------------
[{'AttributeURI': 'http://sandbox.dexpi.org/rdl/DesignHeatFlowRate',
  'Format': 'double',
  'Name': 'DesignHeatFlowRate',
  'Units': 'Kilowatt',
  'UnitsURI': 'http://data.posccaesar.org/rdl/RDS1330919',
  'Value': '313'},
 {'AttributeURI': 'http://sandbox.dexpi.org/rdl/DesignHeatTransferArea',
  'Format': 'double',
  'Name': 'DesignHeatTransferArea',
  'Units': 'MetreSquared',
  'UnitsURI': 'http://data.posccaesar.org/rdl/RDS1358009',
  'Value': '46.8'},
 {'AttributeURI': 'http://sandbox.dexpi.org/rdl/PlateHeight',
  'Format': 'double',
  'Name': 'PlateHeight',
  'Units': 'Millimetre',
  'UnitsURI': 'http://data.posccaesar.org/rdl/RDS1357739',
  'Value': '850'},
 {'AttributeURI': 'http://

## 6. Inspect explicit connections and their endpoints

A `Connection` records `FromID`, `FromNode`, `ToID` and `ToNode`. Keep the component identifiers and port fields intact.

Its parent is useful context: piping links and signal links must not be treated as the same relationship.

**Important:** we are inspecting endpoint ordering. We have not established that every `From` → `To` pair is process-flow direction.


In [79]:
connections = []
for element in live_elements:
    if local_name(element.tag) == "Connection":
        parent = parents[element]
        connections.append({
            "attributes": dict(element.attrib),
            "parent_tag": local_name(parent.tag) if parent is not None else None,
            "parent_id": parent.get("ID") if parent is not None else None,
        })

print("Explicit connections:", len(connections))
print("Parent types:", dict(Counter(c["parent_tag"] for c in connections)))
pprint(connections[:4])


Explicit connections: 29
Parent types: {'PipingNetworkSegment': 23, 'InformationFlow': 6}
[{'attributes': {'FromID': 'FlowInPipeOffPageConnector-1',
                 'FromNode': '1',
                 'ToID': 'Nozzle-1',
                 'ToNode': '1'},
  'parent_id': 'PipingNetworkSegment-1',
  'parent_tag': 'PipingNetworkSegment'},
 {'attributes': {'FromID': 'Nozzle-2',
                 'FromNode': '1',
                 'ToID': 'Nozzle-3',
                 'ToNode': '1'},
  'parent_id': 'PipingNetworkSegment-2',
  'parent_tag': 'PipingNetworkSegment'},
 {'attributes': {'FromID': 'Nozzle-4',
                 'FromNode': '1',
                 'ToID': 'Nozzle-5',
                 'ToNode': '1'},
  'parent_id': 'PipingNetworkSegment-3',
  'parent_tag': 'PipingNetworkSegment'},
 {'attributes': {'FromID': 'Nozzle-6',
                 'FromNode': '1',
                 'ToID': 'SwingCheckValve-1',
                 'ToNode': '2'},
  'parent_id': 'PipingNetworkSegment-4',
  'parent_tag': 'Pipin

## 7. Check duplicate IDs and unresolved component endpoints

Build an index of live XML IDs. A duplicate ID makes reference resolution ambiguous. A connection whose endpoint ID is missing needs investigation.

**Passing this check means only that endpoint IDs exist.** It does not validate endpoint classes, port indices, DEXPI schema conformance or safety.


In [80]:
id_index = defaultdict(list)

for element in live_elements:
    if element.get("ID"):
        id_index[element.get("ID")].append(element)

duplicate_ids = {identifier: [local_name(e.tag) for e in elements]
                 for identifier, elements in id_index.items() if len(elements) > 1}
endpoint_issues = []
for connection in connections:
    attrs = connection["attributes"]

    for field in ("FromID", "ToID"):
        target = attrs.get(field)

        if not target: 
            issue_type = "missing_endpoint"

        elif target not in id_index:
            issue_type = "unresolved_reference"

        elif len(id_index[target]) > 1:
            issue_type = "ambigous_reference"
        else:
            continue

        endpoint_issues.append({
            "issue_type": issue_type,
            "field": field,
            "target": target,
            "parent_id": connection["parent_id"],
            "parent_tag": connection["parent_tag"],
        })

print("Duplicate IDs:", len(duplicate_ids))

print("Endpoint issue types: ")
pprint(dict(Counter(
    issue["issue_type"]
    for issue in endpoint_issues
)))

pprint(endpoint_issues)

Duplicate IDs: 0
Endpoint issue types: 
{'missing_endpoint': 4}
[{'field': 'FromID',
  'issue_type': 'missing_endpoint',
  'parent_id': 'PipingNetworkSegment-20',
  'parent_tag': 'PipingNetworkSegment',
  'target': None},
 {'field': 'ToID',
  'issue_type': 'missing_endpoint',
  'parent_id': 'PipingNetworkSegment-21',
  'parent_tag': 'PipingNetworkSegment',
  'target': None},
 {'field': 'FromID',
  'issue_type': 'missing_endpoint',
  'parent_id': 'PipingNetworkSegment-22',
  'parent_tag': 'PipingNetworkSegment',
  'target': None},
 {'field': 'ToID',
  'issue_type': 'missing_endpoint',
  'parent_id': 'PipingNetworkSegment-23',
  'parent_tag': 'PipingNetworkSegment',
  'target': None}]


## 8. Examine the port declarations for one connection

Look up the first connection's source component, then inspect its direct `ConnectionPoints` child.

Notice that a connection may reference a numeric node field while a declared node also has a string `ID`. We must check the export convention before choosing how to resolve ports; this cell deliberately does not guess that mapping.


In [81]:
if connections:
    first_connection = connections[0]["attributes"]
    source_candidates = id_index.get(first_connection.get("FromID"), [])
    print("Connection:")
    pprint(first_connection)
    for source_element in source_candidates:
        for child in source_element:
            if local_name(child.tag) == "ConnectionPoints":
                print("ConnectionPoints:", dict(child.attrib))
                pprint([dict(node.attrib) for node in child
                        if local_name(node.tag) == "Node"])


Connection:
{'FromID': 'FlowInPipeOffPageConnector-1',
 'FromNode': '1',
 'ToID': 'Nozzle-1',
 'ToNode': '1'}
ConnectionPoints: {'FlowOut': '1', 'NumPoints': '2'}
[{'ID': 'FlowInPipeOffPageConnector-1-DefaultNode'},
 {'ID': 'PipingNode-1', 'Type': 'process'}]


In [82]:
# 8b. Inventory connection endpoints and port declarations
# 8b. Inventory connection endpoints and port declarations

endpoint_records = []

for i, connection in enumerate(connections):
    attrs = connection["attributes"]

    for side in ("From", "To"):
        component_id = attrs.get(f"{side}ID")
        port_ref = attrs.get(f"{side}Node")

        candidates = id_index.get(component_id, [])

        declared_nodes = []

        if len(candidates) == 1:
            component = candidates[0]

            for child in component:
                if local_name(child.tag) != "ConnectionPoints":
                    continue

                for node in child:
                    if local_name(node.tag) == "Node":
                        declared_nodes.append(dict(node.attrib))

        endpoint_records.append({
            "connection_index": i,
            "parent_id": connection["parent_id"],
            "side": side,
            "component_id": component_id,
            "port_ref": port_ref,
            "component_match_count": len(candidates),
            "declared_nodes": declared_nodes,
        })

print("Endpoint records:", len(endpoint_records))

print("Endpoints with missing component IDs:")
pprint([
    r for r in endpoint_records
    if not r["component_id"]
])

print("\nEndpoints without a unique component match:")
pprint([
    r for r in endpoint_records
    if r["component_id"]
    and r["component_match_count"] != 1
])

Endpoint records: 58
Endpoints with missing component IDs:
[{'component_id': None,
  'component_match_count': 0,
  'connection_index': 19,
  'declared_nodes': [],
  'parent_id': 'PipingNetworkSegment-20',
  'port_ref': None,
  'side': 'From'},
 {'component_id': None,
  'component_match_count': 0,
  'connection_index': 20,
  'declared_nodes': [],
  'parent_id': 'PipingNetworkSegment-21',
  'port_ref': None,
  'side': 'To'},
 {'component_id': None,
  'component_match_count': 0,
  'connection_index': 21,
  'declared_nodes': [],
  'parent_id': 'PipingNetworkSegment-22',
  'port_ref': None,
  'side': 'From'},
 {'component_id': None,
  'component_match_count': 0,
  'connection_index': 22,
  'declared_nodes': [],
  'parent_id': 'PipingNetworkSegment-23',
  'port_ref': None,
  'side': 'To'}]

Endpoints without a unique component match:
[]


## 9. Load the file with pyDEXPI — the library is required here

Now use the same mechanism as your original script:

`ProteusSerializer` reads DEXPI into the library's engineering data model. This gives us structured Python objects rather than XML syntax alone.

If the import fails, first check cell 1 and your selected kernel. Do not change or reinstall package versions blindly.


In [83]:
from pydexpi.loaders import ProteusSerializer, MLGraphLoader

serializer = ProteusSerializer()
dexpi_model = serializer.load(str(XML_PATH.parent), XML_PATH.name)

type(dexpi_model).__name__


'DexpiModel'

## 10. Build the graph from the engineering model

Your script uses `parse_dexpi_to_graph`. Newer pyDEXPI APIs may expose `dexpi_to_graph`; this cell checks which method exists and reports the one it uses.

Inspect whether the result is directed and whether it permits multiple edges. A directed graph can still contain containment/reference directions that are not physical flow directions.


In [84]:
loader = MLGraphLoader()
method_name = "parse_dexpi_to_graph"
convert = getattr(loader, method_name, None)
if not callable(convert):
    method_name = "dexpi_to_graph"
    convert = getattr(loader, method_name, None)
if not callable(convert):
    raise RuntimeError("Unsupported GraphLoader API: inspect your installed pyDEXPI version")

graph = convert(dexpi_model)

{"loader_method": method_name,
 "graph_type": type(graph).__name__,
 "directed": graph.is_directed(),
 "multigraph": graph.is_multigraph(),
 "nodes": graph.number_of_nodes(),
 "edges": graph.number_of_edges()}


{'loader_method': 'dexpi_to_graph',
 'graph_type': 'DiGraph',
 'directed': True,
 'multigraph': False,
 'nodes': 33,
 'edges': 36}

## 11. Inspect graph nodes

`graph.nodes(data=True)` returns `(node_id, attributes)` pairs.

Look for `proteusId`, class labels and engineering properties. We inspect these before designing LightRAG entity names. Do not overwrite `label` with a visualization label here; that could hide the original class information.

**Try:** change the slice to inspect more nodes.


In [85]:
node_records = list(graph.nodes(data=True))
node_attribute_keys = Counter(key for _, attributes in node_records for key in attributes)
print("Node attribute keys:", dict(node_attribute_keys))
pprint(node_records[:3])


Node attribute keys: {'tagName': 5, 'tagNamePrefix': 5, 'tagNameSequenceNumber': 5, 'tagNameSuffix': 5, 'equipmentDescription': 5, 'designHeatFlowRate': 2, 'designHeatTransferArea': 2, 'designHeatTransferCoefficient': 2, 'numberOfPlates': 1, 'plateHeight': 1, 'plateWidth': 1, 'dexpi_class': 33, 'temaStandardType': 1, 'designPressureHead': 2, 'designVolumeFlowRate': 2, 'differentialPressure': 2, 'designRotationalSpeed': 2, 'designShaftPower': 2, 'nominalCapacityVolume': 1, 'cylinderLength': 1, 'fluidCode': 19, 'onHold': 19, 'pipingClassArtefact': 19, 'pressureTestCircuitNumber': 19, 'heatTracingType': 18, 'heatTracingTypeRepresentation': 18, 'insulationThickness': 18, 'insulationType': 18, 'lowerLimitHeatTracingTemperature': 18, 'numberOfPorts': 9, 'operation': 9, 'pipingClassCode': 18, 'pipingComponentName': 18, 'pipingComponentNumber': 18, 'flowInPipingClassCode': 1, 'flowOutPipingClassCode': 1, 'locationRegistrationNumber': 1, 'positionNumber': 1, 'setPressureHigh': 1, 'setPressureLo

## 12. Inspect graph edges and their relationship meanings

For a multigraph, the edge key distinguishes parallel edges. Preserve it when inspecting or exporting.

Your RDF exporter used `label` and `attr_name` to decide predicates. We inspect those values directly instead of immediately turning all edges into `connected_to`.

The pyDEXPI edge count is not expected to equal the XML `Connection` count: the model graph also contains composition and other references.


In [86]:
# 12. Inspect graph edges and relationship meanings
edge_records = []

if graph.is_multigraph():
    for u, v, key in graph.edges(keys=True):
        edge_data = graph.get_edge_data(u, v, key)

        edge_records.append({
            "source": u,
            "target": v,
            "key": key,
            "attributes": dict(edge_data),
        })
else:
    for u, v in graph.edges():
        edge_data = graph.get_edge_data(u, v)

        edge_records.append({
            "source": u,
            "target": v,
            "attributes": dict(edge_data),
        })

# Count actual relationship types
relationship_counts = Counter(
    e["attributes"].get("dexpi_class", "<missing>")
    for e in edge_records
)

print("Relationship types:")
pprint(dict(relationship_counts))

# Regression checks
assert len(edge_records) == graph.number_of_edges()

assert all(
    isinstance(e["attributes"], dict)
    for e in edge_records
)

assert all(
    "dexpi_class" in e["attributes"]
    for e in edge_records
), "Edge relationship classes are missing!"

print("Edge inspection passed.")

Relationship types:
{'DirectPipingConnection': 2,
 'MeasuringLineFunction': 3,
 'OperatedValveReference': 3,
 'Pipe': 25,
 'SignalConveyingFunction': 3}
Edge inspection passed.


## 13. Trace one XML equipment ID into the graph

Start with the selected source equipment. Try the `proteusId` attribute used by your original script, or an identical graph node ID. Inspect incoming and outgoing edges separately.

An empty match is a finding to investigate, not proof that the equipment was lost: this pyDEXPI version may expose the source ID differently.


In [87]:
xml_equipment_id = selected_equipment.get("ID")
xml_component_class = selected_equipment.get("ComponentClass")

print ("XML ID: ", xml_equipment_id)
print("XML Class: ", xml_component_class)

# Exact identitty matching only
exact_matches = [node_id for node_id, attributes in node_records
           if xml_equipment_id is not None and
           (attributes.get("proteusId") == xml_equipment_id
            or str(node_id) == xml_equipment_id)]

print("\nExact identity matches:")
pprint(exact_matches)

# Diagnostic candidates, NOT confirmed identity matches
equipment_classes = {
    "PlateHeatExchanger",
    "TubularHeatExchanger",
    "CentrifugalPump",
    "ReciprocatingPump",
    "Tank",
}

candidates = [
    {
        "graph_id": node_id,
        "dexpi_class": attributes.get("dexpi_class"),
        "tagName": attributes.get("tagName"),
    }
    for node_id, attributes in node_records
    if attributes.get("dexpi_class") in equipment_classes
]

pprint(candidates)

if not exact_matches:
        print(
        "\nNo exact ID mapping found. "
        "Candidates must not be treated as confirmed matches."
    )

XML ID:  PlateHeatExchanger-1
XML Class:  PlateHeatExchanger

Exact identity matches:
[]
[{'dexpi_class': 'PlateHeatExchanger',
  'graph_id': '6354052e-c4b5-48fb-a314-01d28b76b887',
  'tagName': 'H1007'},
 {'dexpi_class': 'TubularHeatExchanger',
  'graph_id': '79c3d992-fb34-4fce-9884-b957859d1718',
  'tagName': 'H1008'},
 {'dexpi_class': 'CentrifugalPump',
  'graph_id': 'f69ae701-edcd-48a5-8d93-d139b7c1e714',
  'tagName': 'P4711'},
 {'dexpi_class': 'ReciprocatingPump',
  'graph_id': '48b671f2-f685-4f6f-8d4a-85a86411d4a8',
  'tagName': 'P4712'},
 {'dexpi_class': 'Tank',
  'graph_id': '0b96c24f-2327-44b3-bd02-bfa9b8bbc523',
  'tagName': 'T4750'}]

No exact ID mapping found. Candidates must not be treated as confirmed matches.


## 14. Save an inspection report

This writes a JSON report we can review together. Non-JSON Python values are converted to their `repr` for inspection. It is **not a lossless serialization** and must not be used as the authoritative model or imported into LightRAG as-is.

The report contains the full node/edge inspection records; the original XML stays unchanged.


In [88]:
import json

report = {
    "source_file": XML_PATH.name,
    "pydexpi_version": version("pydexpi"),
    "xml": {
        "live_tag_counts": dict(tag_counts),
        "catalogue_element_count": catalogue_element_count,
        "connection_count": len(connections),
        "connections": connections,
        "duplicate_ids": duplicate_ids,
        "endpoint_issues": endpoint_issues,
    },
    "graph": {
        "loader_method": method_name,
        "graph_type": type(graph).__name__,
        "directed": graph.is_directed(),
        "multigraph": graph.is_multigraph(),
        "node_count": graph.number_of_nodes(),
        "edge_count": graph.number_of_edges(),
        "relationship_signatures": [
            {"dexpi_class": relation_type, "count": count,}
            for relation_type, count in relationship_counts.items()
        ],
        "nodes": [{"id": node_id, "attributes": attributes}
                  for node_id, attributes in node_records],
        "edges": edge_records,
    },
}

report["validation"] = {
    "duplicate_id_count": len(duplicate_ids),
    "endpoint_issues_counts": dict(
        Counter(
            issue["issue_type"]
            for issue in endpoint_issues
        )
    ),
    "relation_type_counts": dict(
        relationship_counts
    ),
    "graph_nodes_without_proteus_id": sum(
        "proteusId" not in attrs
        for _, attrs in node_records
    ),

    "xml_equipment_to_graph_identity": {
        "selected_xml_id": xml_equipment_id,
        "exact_matches": [
            str(node_id) for node_id in exact_matches
        ],
    },
    "port_validation_status": "not_verified",
    "flow_direction_validation_status": "not_verified",
    "xml_graph_equivalence_status": "not_verified",
}

report["xml"]["endpoint_records"] = endpoint_records

OUTPUT_DIR = PROJECT_ROOT / "output" / "dexpi_notebook_inspection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / "inspection.json"
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=repr),
                       encoding="utf-8")
print("Saved:", REPORT_PATH)


Saved: /home/duc/Desktop/raganything_hazop/output/dexpi_notebook_inspection/inspection.json


In [91]:
print("Script:", SCRIPT_PATH)
print("Script exists:", SCRIPT_PATH.is_file())

print("XML:", XML_PATH)
print("XML exists:", XML_PATH.is_file())

print("Output directory:", OUTPUT_DIR)

Script: /home/duc/Desktop/raganything_hazop/tests/dexpi_port_flow_inspection.py
Script exists: True
XML: /home/duc/Desktop/raganything_hazop/data/others/C01V04-VER.EX01.xml
XML exists: True
Output directory: /home/duc/Desktop/raganything_hazop/output/dexpi_notebook_inspection


## 15. Validate port references and inspect flow-direction evidence

In [99]:
import subprocess
import sys
import json

SCRIPT_PATH = (
    PROJECT_ROOT
    / "tests"
    / "dexpi_port_flow_inspection.py"
)

PORT_REPORT_PATH = (
    OUTPUT_DIR
    / "port_flow_inspection.json"

)


assert SCRIPT_PATH.is_file(), f"Script not found: {SCRIPT_PATH}"
assert XML_PATH.is_file(), f"XML not found: {XML_PATH}"

# Execute using the active notebook kernel's Python.
result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT_PATH),
        str(XML_PATH),
        "--output",
        str(PORT_REPORT_PATH),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print("Return code:", result.returncode)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

if result.returncode != 0:
    raise RuntimeError("Port/flow inspection failed. See STDERR above.")


with PORT_REPORT_PATH.open(encoding="utf-8") as f:
    port_report = json.load(f)

port_report["summary"]

Return code: 0
STDOUT: {
  "shape_catalogue_elements_excluded": 459,
  "connection_count": 29,
  "endpoint_count": 58,
  "endpoint_status_counts": {
    "index_resolved": 54,
    "missing_endpoint": 4
  },
  "endpoint_type_counts": {
    "process": 42,
    "not_checked": 4,
    "signal": 12
  },
  "duplicate_id_count": 0,
  "pipe_flow_arrow_count": 8,
  "direction_hint_count": 21
}
Saved: /home/duc/Desktop/raganything_hazop/output/dexpi_notebook_inspection/port_flow_inspection.json

STDERR: 


{'shape_catalogue_elements_excluded': 459,
 'connection_count': 29,
 'endpoint_count': 58,
 'endpoint_status_counts': {'index_resolved': 54, 'missing_endpoint': 4},
 'endpoint_type_counts': {'process': 42, 'not_checked': 4, 'signal': 12},
 'duplicate_id_count': 0,
 'pipe_flow_arrow_count': 8,
 'direction_hint_count': 21}

## 16. Review issues requiring manual investigation

In [100]:
from pprint import pprint

# Port references not resolved by numeric index
port_issues = [
    result
    for result in port_report["endpoint_results"]
    if result["status"] != "index_resolved"
]

print("Port issues:")
pprint(port_issues)

# Inspect all detected flow arrows
print("\nFlow arrows:")
for arrow in port_report["flow_arrows"]:
    print({
        "id": arrow["id"],
        "segment": arrow["containing_segment_id"],
        "ancestors": arrow["ancestors"],
    })

# Direction-related XML attributes
print("\nDirection metadata:")
pprint(port_report["direction_hints"])

Port issues:
[{'component_id': None,
  'connection_attributes': {'ToID': 'Nozzle-13', 'ToNode': '1'},
  'connection_index': 19,
  'declared_nodes': [],
  'node_type': None,
  'parent_id': 'PipingNetworkSegment-20',
  'parent_tag': 'PipingNetworkSegment',
  'port_index_convention': 'zero_based_all_direct_Node_children',
  'port_ref': None,
  'resolved_node': None,
  'side': 'From',
  'status': 'missing_endpoint',
  'type_consistency': 'not_checked'},
 {'component_id': None,
  'connection_attributes': {'FromID': 'Nozzle-14', 'FromNode': '1'},
  'connection_index': 20,
  'declared_nodes': [],
  'node_type': None,
  'parent_id': 'PipingNetworkSegment-21',
  'parent_tag': 'PipingNetworkSegment',
  'port_index_convention': 'zero_based_all_direct_Node_children',
  'port_ref': None,
  'resolved_node': None,
  'side': 'To',
  'status': 'missing_endpoint',
  'type_consistency': 'not_checked'},
 {'component_id': None,
  'connection_attributes': {'ToID': 'Nozzle-15', 'ToNode': '1'},
  'connection_

## Stop here before the LightRAG adapter

Review these questions:
1. Which fields identify the equipment and its class?
2. Which edges describe containment, and which describe connections or signals?
3. Can we trace connection endpoints and ports from XML into the model graph?
4. Which direction represents containment/reference direction, and which has evidence of process flow?

Send the `inspection.json` report and any cell errors. Our next notebook step will define component identities and the LightRAG adapter **using the graph we actually observed**.

References:
- [pyDEXPI source and documentation](https://github.com/process-intelligence-research/pyDEXPI)
- [uv and Jupyter project environments](https://docs.astral.sh/uv/guides/integration/jupyter/)
